# Custom Backend — DynamoDB Session Repository

The built-in `FileSessionManager` and `S3SessionManager` cover common cases,
but you can implement any storage backend by extending `SessionRepository` and
wrapping it with `RepositorySessionManager`.

This notebook implements a DynamoDB-backed session repository.

## DynamoDB Table Schema

| Attribute | Type | Description |
|-----------|------|-------------|
| `pk` | String (Partition Key) | `SESSION#<session_id>` |
| `sk` | String (Sort Key) | `META`, `AGENT#<agent_id>`, or `MSG#<agent_id>#<message_id>` |
| `data` | Map | The serialized session, agent, or message data |

In [ ]:
%pip install -q --upgrade strands-agents boto3

## Create the DynamoDB table

Create a table with a composite primary key (`pk` + `sk`).

In [ ]:
import boto3

TABLE_NAME = "strands-sessions-tutorial"

dynamodb = boto3.client("dynamodb")

try:
    dynamodb.create_table(
        TableName=TABLE_NAME,
        KeySchema=[
            {"AttributeName": "pk", "KeyType": "HASH"},
            {"AttributeName": "sk", "KeyType": "RANGE"},
        ],
        AttributeDefinitions=[
            {"AttributeName": "pk", "AttributeType": "S"},
            {"AttributeName": "sk", "AttributeType": "S"},
        ],
        BillingMode="PAY_PER_REQUEST",
    )
    waiter = dynamodb.get_waiter("table_exists")
    waiter.wait(TableName=TABLE_NAME)
    print(f"Table '{TABLE_NAME}' created.")
except dynamodb.exceptions.ResourceInUseException:
    print(f"Table '{TABLE_NAME}' already exists.")

## Implement the DynamoDB SessionRepository

In [ ]:
from typing import Any

import boto3
from strands.session.session_repository import SessionRepository
from strands.types.session import Session, SessionAgent, SessionMessage


class DynamoDBSessionRepository(SessionRepository):
    """DynamoDB-backed session repository using a single-table design."""

    def __init__(self, table_name: str):
        self.table = boto3.resource("dynamodb").Table(table_name)

    def _session_pk(self, session_id: str) -> str:
        return f"SESSION#{session_id}"

    def create_session(self, session: Session, **kwargs: Any) -> Session:
        self.table.put_item(
            Item={"pk": self._session_pk(session.session_id), "sk": "META", "data": session.to_dict()}
        )
        return session

    def read_session(self, session_id: str, **kwargs: Any) -> Session | None:
        resp = self.table.get_item(Key={"pk": self._session_pk(session_id), "sk": "META"})
        item = resp.get("Item")
        if not item:
            return None
        return Session.from_dict(item["data"])

    def delete_session(self, session_id: str, **kwargs: Any) -> None:
        pk = self._session_pk(session_id)
        # Note: For production use, handle DynamoDB pagination for sessions with many items
        # Query all items for this session and delete them
        resp = self.table.query(KeyConditionExpression=boto3.dynamodb.conditions.Key("pk").eq(pk))
        with self.table.batch_writer() as batch:
            for item in resp.get("Items", []):
                batch.delete_item(Key={"pk": item["pk"], "sk": item["sk"]})

    def create_agent(self, session_id: str, session_agent: SessionAgent, **kwargs: Any) -> None:
        self.table.put_item(
            Item={
                "pk": self._session_pk(session_id),
                "sk": f"AGENT#{session_agent.agent_id}",
                "data": session_agent.to_dict(),
            }
        )

    def read_agent(self, session_id: str, agent_id: str, **kwargs: Any) -> SessionAgent | None:
        resp = self.table.get_item(
            Key={"pk": self._session_pk(session_id), "sk": f"AGENT#{agent_id}"}
        )
        item = resp.get("Item")
        if not item:
            return None
        return SessionAgent.from_dict(item["data"])

    def update_agent(self, session_id: str, session_agent: SessionAgent, **kwargs: Any) -> None:
        self.create_agent(session_id, session_agent, **kwargs)

    def create_message(
        self, session_id: str, agent_id: str, session_message: SessionMessage, **kwargs: Any
    ) -> None:
        self.table.put_item(
            Item={
                "pk": self._session_pk(session_id),
                "sk": f"MSG#{agent_id}#{int(session_message.message_id):010d}",
                "data": session_message.to_dict(),
            }
        )

    def read_message(
        self, session_id: str, agent_id: str, message_id: int, **kwargs: Any
    ) -> SessionMessage | None:
        resp = self.table.get_item(
            Key={"pk": self._session_pk(session_id), "sk": f"MSG#{agent_id}#{int(message_id):010d}"}
        )
        item = resp.get("Item")
        if not item:
            return None
        return SessionMessage.from_dict(item["data"])

    def update_message(
        self, session_id: str, agent_id: str, session_message: SessionMessage, **kwargs: Any
    ) -> None:
        self.create_message(session_id, agent_id, session_message, **kwargs)

    def list_messages(
        self,
        session_id: str,
        agent_id: str,
        limit: int | None = None,
        offset: int = 0,
        **kwargs: Any,
    ) -> list[SessionMessage]:
        from boto3.dynamodb.conditions import Key

        # Note: For production use, handle DynamoDB pagination for large message histories
        resp = self.table.query(
            KeyConditionExpression=Key("pk").eq(self._session_pk(session_id))
            & Key("sk").begins_with(f"MSG#{agent_id}#"),
        )
        messages = [SessionMessage.from_dict(item["data"]) for item in resp.get("Items", [])]
        messages.sort(key=lambda m: m.message_id)

        # Apply offset and limit
        offset = int(offset)
        messages = messages[offset:]
        if limit is not None:
            messages = messages[:int(limit)]
        return messages


print("DynamoDBSessionRepository defined.")

## Use the custom repository with RepositorySessionManager

In [ ]:
from strands import Agent
from strands.session.repository_session_manager import RepositorySessionManager

SESSION_ID = "dynamodb-demo-session"

repo = DynamoDBSessionRepository(table_name=TABLE_NAME)
session_manager = RepositorySessionManager(session_id=SESSION_ID, session_repository=repo)

agent = Agent(
    system_prompt="You are a helpful assistant. Remember details the user shares with you.",
    session_manager=session_manager,
)

response = agent("I'm building a real-time analytics dashboard using Kinesis and OpenSearch.")
print(response)

In [ ]:
# Restore from DynamoDB — new agent, same session
restored_repo = DynamoDBSessionRepository(table_name=TABLE_NAME)
restored_sm = RepositorySessionManager(session_id=SESSION_ID, session_repository=restored_repo)

restored_agent = Agent(
    system_prompt="You are a helpful assistant. Remember details the user shares with you.",
    session_manager=restored_sm,
)

response = restored_agent("What AWS services am I using for my project?")
print(response)

## Cleanup

In [ ]:
dynamodb_client = boto3.client("dynamodb")
dynamodb_client.delete_table(TableName=TABLE_NAME)
print(f"Table '{TABLE_NAME}' deleted.")